# Probing a Model

This notebook shows how to evaluate a model's embeddings with **linear probing**.
After generating embeddings with a pretrained backbone, we train a small linear
classifier (a *probe*) on top of the *frozen* embeddings to measure how well your
own ground-truth species can be linearly separated. We then load the trained
probe and apply it to new embeddings to generate predictions.


Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.

---
## 1. Setup & Configuration
Import necessary modules 

In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

/home/siriussound/Code/testing_repos/bacpipe/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/siriussound/Code/testing_repos/bacpipe/bacpipe/embedding_evaluation/visualization/dashboard.py:42: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("plotly")


Set the working directory to the repository root and clean the previous tests if needed/wanted

In [2]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')
print(os.listdir('.'))


# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'probing_a_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

['.vscode', 'bacpipe_results', 'run_pipeline.py', 'embed_obj = bacpipe.py', '.gitignore', 'bacpipe', '.readthedocs.yml', '.python-version', '.pytest_cache', 'pyproject.toml', 'bacpipe_model_checkpoints', '.github', '.coverage', '.venv', 'LICENSE', 'requirements_no_cuda.txt', 'requirements_tf_gpu.txt', '.mypy_cache', 'env_build', '.gitattributes', 'requirements_no_tf.txt', '.pre-commit-config.yaml', 'src', 'dist', '.git', 'README.md', 'docs', 'uv.lock']
Folder bacpipe_results/probing_a_model not found.


Set the global constants used throughout the notebook.

In [3]:
MODEL_NAME = 'perch_v2'                  # name of the model to run. Supported models are in bacpipe.supported_models
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

---
## 2. Probing Pipeline

`bacpipe.run_pipeline_for_single_model` generates the embeddings for the chosen
model and dataset. `bacpipe.ground_truth_by_model` must be run *after* the
embeddings exist: it reads `annotations.csv`, aligns the annotations to the
model's time grid, and saves a `ground_truth.npy` file linking each embedding to
its labels.

`bacpipe.probing_pipeline` then trains a linear probe on top of the frozen
embeddings (using the train/validation/test split defined in
`bacpipe.settings.probe_configs`) and evaluates it on the held-out test set.
Because the backbone weights stay frozen, the probe scores measure how
*linearly separable* the species are in the model's embedding space: high scores
mean the embeddings already separate the species well, low scores mean they do
not (for this linear read-out).

The function returns:

- `probe`: the trained linear probe model,
- `label2idx`: a dictionary mapping each class label to the column index in the
  probe's prediction array,
- `metrics`: a dictionary with the evaluation results, including the overall
  scores (`macro_accuracy`, `micro_accuracy`, `auc`, `macro_f1`, `micro_f1`) and
  the per-class accuracy.


In [4]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
).embeddings(return_type='array')

# Run this function after computing the embeddings otherwise it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR, 
    annotations_filename='annotations.csv',
    overwrite=False
)

probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds)


Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

perch_v2 checkpoint exists.




###### Generating embeddings using PERCH_V2 ######

finding audio files: 11it [00:00, 19912.54it/s]
Found 7 number of audio files.
2026-08-26 16:52:12.729067066 [W:onnxruntime:Default, device_discovery.cc:285 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card5": device_discovery.cc:94 ReadFileContents Failed to open file: "/sys/class/drm/card5/device/vendor"
2026-08-26 16:52:12.729717809 [W:onnxruntime:Default, device_discovery.cc:285 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card3": device_discovery.cc:94 ReadFileContents Failed to open file: "/sys/class/drm/card3/device/vendor"
2026-08-26 16:52:12.730187955 [W:onnxruntime:Default, device_discovery.cc:285 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card4": device_discovery.cc:94 ReadFileContents Failed to open file: "/sys/class/drm/card4/device/vendor"

perch_v2 initialized using providers: ['CPUExecutionProvider']


getting time of day: 100%|██████████| 7/7 [00:00<00:00, 161319.38it/s]
getting time per embeddings: 7it [00:00, 62203.66it/s]
getting day of year: 100%|██████████| 7/7 [00:00<00:00, 190650.18it/s]
getting continuous timestamps: 7it [00:00, 51599.52it/s]
getting parent directory: 7it [00:00, 36247.07it/s]
getting audio file names: 7it [00:00, 55188.21it/s]
Building metadata labels: 100%|██████████| 7/7 [00:00<00:00, 330.97it/s]
Found 19 samples in the train set with 3 unique labels.
Epoch 1/20
Epoch [1/20], Loss: 1.0794, Accuracy: 68.42%
Epoch 2/20
Epoch [2/20], Loss: 1.0509, Accuracy: 84.21%
Epoch 3/20
Epoch [3/20], Loss: 1.0229, Accuracy: 84.21%
Epoch 4/20
Epoch [4/20], Loss: 0.9956, Accuracy: 84.21%
Epoch 5/20
Epoch [5/20], Loss: 0.9690, Accuracy: 84.21%
Epoch 6/20
Epoch [6/20], Loss: 0.9431, Accuracy: 84.21%
Epoch 7/20
Epoch [7/20], Loss: 0.9178, Accuracy: 84.21%
Epoch 8/20
Epoch [8/20], Loss: 0.8931, Accuracy: 84.21%
Epoch 9/20
Epoch [9/20], Loss: 0.8692, Accuracy: 84.21%
Epoch 10/

In [5]:
print("======== display probe ==========")
display(type(probe))
print("======== display label2idx ======")
display(label2idx)
print("======== display metrics ========")
display(metrics)

======== display probe ==========


bacpipe.embedding_evaluation.probing.train_probe.LinearProbe

======== display label2idx ======


{'Common Chaffinch': 0, 'Common Cuckoo': 1, 'Eurasian Blackbird': 2}

======== display metrics ========


{'overall': {'macro_accuracy': 0.6666666666666666,
  'auc': 1.0,
  'macro_f1': 0.5555555555555555},
 'items_per_class': {'Common Chaffinch': 2,
  'Common Cuckoo': 2,
  'Eurasian Blackbird': 3},
 'per_class_accuracy': {'Common Chaffinch': 1.0,
  'Common Cuckoo': 0.0,
  'Eurasian Blackbird': 1.0},
 'config': {'main_results_dir': 'bacpipe_results/probing_a_model',
  'embed_parent_dir': 'embeddings',
  'dim_reduc_parent_dir': 'dim_reduced_embeddings',
  'evaluations_dir': 'evaluations',
  'model_base_path': 'bacpipe_model_checkpoints',
  'global_batch_size': 8,
  'audio_suffixes': ['.wav', '.WAV', '.aif', '.mp3', '.MP3', '.flac', '.ogg'],
  'padding': 'wrap',
  'avoid_pipelined_gpu_inference': False,
  'nr_parallel_workers': False,
  'rm_embedding_on_keyboard_interrupt': False,
  'check_if_already_processed': True,
  'check_if_already_dim_reduced': True,
  'annotations_filename': 'annotations.csv',
  'only_embed_annotations': False,
  'min_annotation_length': 0,
  'metadata_label_keys': ['

---
## 3. Probe Inference

`bacpipe.prepare_probe_inference` loads a probe that was trained and saved by the
probing pipeline, together with the `label2index.json` file describing the
class-to-column mapping. The returned probe is restored to its exact post-training
state and is ready for inference.

`bacpipe.run_probe_inference` applies that probe to embeddings and returns the
predictions. With `return_binary_presence=False` it returns the softmax
probabilities for each class; with `return_binary_presence=True` (the default) it
thresholds the probabilities with `threshold` (default `0.5`) and returns a binary
presence matrix.


In [6]:
probe, label2idx = bacpipe.prepare_probe_inference(model=MODEL_NAME)

predictions = bacpipe.run_probe_inference(
    model=MODEL_NAME,
    linear_probe=probe,
    threshold=0.5,
    embeds=embeds,
    return_binary_presence=False
)

display(predictions)


array([[0.5006397 , 0.23082861, 0.26853174],
       [0.5205781 , 0.21514112, 0.26428077],
       [0.52853525, 0.21022831, 0.2612365 ],
       [0.58537555, 0.17141691, 0.24320751],
       [0.5531739 , 0.1902108 , 0.25661537],
       [0.5373278 , 0.20748387, 0.25518826],
       [0.57240653, 0.17982683, 0.24776666],
       [0.5812186 , 0.18572831, 0.23305312],
       [0.5487751 , 0.20639034, 0.24483462],
       [0.56728834, 0.18744011, 0.24527156],
       [0.52262354, 0.21649861, 0.26087794],
       [0.49602687, 0.23903045, 0.2649427 ],
       [0.5312988 , 0.21161191, 0.2570893 ],
       [0.35007608, 0.361266  , 0.2886579 ],
       [0.3627609 , 0.3438349 , 0.29340413],
       [0.40429318, 0.30798402, 0.28772286],
       [0.38031346, 0.27533814, 0.34434843],
       [0.37448975, 0.3219713 , 0.30353886],
       [0.40244985, 0.3154602 , 0.28209004],
       [0.19956204, 0.15961121, 0.64082676],
       [0.21909761, 0.19205953, 0.58884287],
       [0.19719101, 0.17894682, 0.62386215],
       [0.